# 06 — PyTorch Tensors & Autograd

## Why PyTorch After NumPy?

In the previous notebooks we computed gradients by hand — tedious and  
error-prone for big networks. **PyTorch** automates gradient computation  
while feeling almost identical to NumPy for basic operations.

The key addition is **autograd**: PyTorch tracks every operation on a tensor  
and can automatically differentiate through the whole computation graph.

We'll cover:
1. Tensor basics (NumPy you already know)
2. Autograd — automatic gradient computation
3. Linear regression using autograd (no hand-derived formulas!)
4. GPU support

## Part 1 — Tensor Basics

A PyTorch `Tensor` behaves just like a NumPy array.  Most operations  
are identical; the names just changed slightly.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f'a         = {a}')
print(f'b         = {b}')
print(f'a + b     = {a + b}')
print(f'a * b     = {a * b}')
print(f'a.mean()  = {a.mean()}')

matrix = torch.zeros(3, 4)
print(f'\ntorch.zeros(3, 4): shape = {matrix.shape}')
print(matrix)

In [ ]:
# NumPy ↔ PyTorch interop (they share memory — zero copy!)
np_array = np.array([1.0, 2.0, 3.0])
tensor   = torch.from_numpy(np_array)
back     = tensor.numpy()

print(f'NumPy array : {np_array}')
print(f'As tensor   : {tensor}')
print(f'Back to NumPy: {back}')
print()
print('They share memory — modifying one changes the other:')
np_array[0] = 99.0
print(f'  np_array[0] = 99, tensor = {tensor}')

## Part 2 — Autograd: Automatic Gradient Computation

Mark a tensor with `requires_grad=True` and PyTorch will track every  
operation on it.  When you call `.backward()`, it computes all gradients  
automatically via the chain rule.

**Example:** `y = w·x² + 5`  
The derivative with respect to `w` is: `dy/dw = x²`

In [ ]:
w = torch.tensor(2.0, requires_grad=True)   # parameter we want to differentiate
x = torch.tensor(3.0)                        # input (fixed)

y = w * x**2 + 5

print(f'y = w * x² + 5  where w={w.item()}, x={x.item()}')
print(f'y = {y.item()}')

# Ask PyTorch to compute dy/dw
y.backward()

print(f'\ndy/dw computed by PyTorch : {w.grad.item()}')
print(f'dy/dw by hand (= x² = 9) : 9.0')

## Part 3 — Linear Regression with Autograd

Previously we derived the gradient formulas for `w` and `b` by hand.  
Now PyTorch does it for us — we just define the loss and call `.backward()`.

Same dataset as notebook 02.

In [ ]:
np.random.seed(42)
x_np = np.linspace(0, 10, 30)
y_np = 10 * x_np + 30 + np.random.randn(30) * 8

x_t = torch.tensor(x_np, dtype=torch.float32)
y_t = torch.tensor(y_np, dtype=torch.float32)

# Visualise the data
plt.figure(figsize=(7, 4))
plt.scatter(x_np, y_np, color="steelblue", label="data")
plt.xlabel("x"); plt.ylabel("y")
plt.title("Dataset for linear regression")
plt.tight_layout()
plt.show()

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.01
losses = []

for step in range(200):
    # Forward pass
    y_pred = w * x_t + b
    loss   = ((y_pred - y_t) ** 2).mean()

    # Backward pass — PyTorch computes dLoss/dw and dLoss/db
    loss.backward()

    # Update — we must tell PyTorch NOT to track these operations
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # Clear gradients (they accumulate by default)
    w.grad.zero_()
    b.grad.zero_()

    losses.append(loss.item())

print(f'Learned : y = {w.item():.2f} * x + {b.item():.2f}')
print(f'Truth   : y = 10.00 * x + 30.00')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Fitted line
ax1.scatter(x_np, y_np, color="steelblue", label="data")
x_line = np.array([0.0, 10.0])
y_line = w.item() * x_line + b.item()
ax1.plot(x_line, y_line, color="crimson", linewidth=2, label="fitted line")
ax1.set_xlabel("x"); ax1.set_ylabel("y")
ax1.set_title(f"Result: y = {w.item():.1f}x + {b.item():.1f}")
ax1.legend()

# Loss curve
ax2.plot(losses, color="darkorange")
ax2.set_xlabel("step"); ax2.set_ylabel("MSE loss")
ax2.set_title("Loss curve")

plt.tight_layout()
plt.show()

## Part 4 — GPU Support

One of PyTorch's biggest advantages: move tensors to the GPU with `.to(device)`  
and the exact same code runs (potentially 100× faster) on GPU hardware.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Device available: {device}')

# To run on GPU (if available):
# x_t = x_t.to(device)
# y_t = y_t.to(device)
# w   = torch.tensor(0.0, requires_grad=True, device=device)
# ... rest of the code is identical

if device == "cpu":
    print("No GPU found — that's fine, all our examples are tiny.")
else:
    print("GPU ready! Real models can train dramatically faster on CUDA.")

## Key Lessons

1. **Tensors** are almost identical to NumPy arrays — switch costs are low.
2. **`requires_grad=True`** tells PyTorch to track operations for differentiation.
3. **`.backward()`** computes all gradients automatically — no hand-derived formulas.
4. **`torch.no_grad()`** prevents PyTorch from tracking the weight-update step.
5. **GPU acceleration** is one `.to('cuda')` call away.